In [ ]:
from pathlib import Path
import csv
import sys
sys.dont_write_bytecode = True

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "data").is_dir()),
    None,
)

CSV_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "resultados"

summary_path = CSV_DIR / "resumen_simulacion.csv"

with summary_path.open("r", newline="", encoding="utf-8") as file:
    summary_rows = csv.DictReader(file)
    simulation_parameters = {row["metrica"]: row["valor"] for row in summary_rows}

L = float(simulation_parameters["l"])
MAX_FRAMES = 240  # Reduce este valor si la animación consume demasiada memoria.
INTERVAL_MS = 40

trajectory_path = CSV_DIR / "simulacionParticulas.csv"
trajectory = np.loadtxt(trajectory_path, delimiter=",", skiprows=1)
if trajectory.ndim == 1:
    trajectory = trajectory.reshape(1, -1)

times, frame_starts = np.unique(trajectory[:, 0], return_index=True)
frame_stops = np.r_[frame_starts[1:], len(trajectory)]
frame_sizes = frame_stops - frame_starts

particle_count = int(frame_sizes[0])

frame_indices = np.unique(np.linspace(0, len(frame_starts) - 1,
                                      min(len(frame_starts), MAX_FRAMES), dtype=int))
frames = [trajectory[frame_starts[index]:frame_stops[index], 1:5]
          for index in frame_indices]
animation_times = times[frame_indices]
print(f"{particle_count} partículas, método={simulation_parameters['metodo']}, "
      f"epsilon={simulation_parameters['epsilon']:g}, sigma={simulation_parameters['sigma']:g}, "
      f"dt={simulation_parameters['dt']:g}, tf={simulation_parameters['tf']:g}, l={L:g}; "
      f"{len(times)} configuraciones, t ∈ [{times[0]:g}, {times[-1]:g}]")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
ax.set(xlim=(0, L), ylim=(0, L), xlabel="x", ylabel="y",
       title=f"Partículas — {simulation_parameters['metodo']}, "
             f"epsilon={simulation_parameters['epsilon']:g}, sigma={simulation_parameters['sigma']:g}, l={L:g}")
ax.set_aspect("equal")
ax.grid(alpha=0.2)
scatter = ax.scatter(frames[0][:, 0], frames[0][:, 1], s=12, c="#1769aa", alpha=0.8)
arrow_stride = max(1, particle_count // 250)
initial_velocity = frames[0][::arrow_stride, 2:4]
initial_magnitude = np.linalg.norm(initial_velocity, axis=1)
initial_direction = initial_velocity / np.maximum(initial_magnitude[:, None], 1e-12) * 0.7
arrows = ax.quiver(frames[0][::arrow_stride, 0], frames[0][::arrow_stride, 1],
                   initial_direction[:, 0], initial_direction[:, 1], color="#e4572e", angles="xy",
                   scale_units="xy", scale=1, width=0.0025)
time_label = ax.text(0.02, 0.98, "", transform=ax.transAxes,
                     va="top", bbox={"facecolor": "white", "alpha": 0.8})

def update(frame_number):
    state = frames[frame_number]
    xy = state[:, :2]
    velocity = state[::arrow_stride, 2:4]
    magnitude = np.linalg.norm(velocity, axis=1)
    direction = velocity / np.maximum(magnitude[:, None], 1e-12) * 0.7
    scatter.set_offsets(xy)
    arrows.set_offsets(xy[::arrow_stride])
    arrows.set_UVC(direction[:, 0], direction[:, 1])
    time_label.set_text(f"t = {animation_times[frame_number]:.5g}")
    return scatter, arrows, time_label

animation = FuncAnimation(fig, update, frames=len(frames),
                          interval=INTERVAL_MS, blit=False, repeat=False)
animation.save(FIG_DIR / "simulacion.gif", writer="pillow", fps=1000 // INTERVAL_MS, dpi=80)
plt.close(fig)
display(HTML(animation.to_jshtml(default_mode="once")))

In [ ]:
mean_speed = []
max_speed = []

for start, stop in zip(frame_starts, frame_stops):
    velocities = trajectory[start:stop, 3:5]
    speeds = np.hypot(velocities[:, 0], velocities[:, 1])
    mean_speed.append(speeds.mean())
    max_speed.append(speeds.max())
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(times, mean_speed, label="Rapidez media")
ax.plot(times, max_speed, label="Rapidez máxima", alpha=0.8)
ax.set(xlabel="Tiempo", ylabel="Rapidez",
       title=f"Evolución de velocidades — {simulation_parameters['metodo']}, "
             f"epsilon={simulation_parameters['epsilon']:g}, sigma={simulation_parameters['sigma']:g}, "
             f"dt={simulation_parameters['dt']:g}, tf={simulation_parameters['tf']:g}")
ax.grid(alpha=0.25)
ax.legend()
fig.savefig(FIG_DIR / "evolucion_rapidez.png", dpi=160, bbox_inches="tight")
plt.show()